# Utility Functions

> Functions of various forms

In [ ]:
#| default_exp utils

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, math, pandas as pd, numpy as np, multiprocessing as mp, zarr, dask.array as da

In [ ]:
#| export
def _no_grad_trunc_normal_(tensor, mean, std, a, b):
    """
    Cut & paste from PyTorch official master until it's in a few official releases 
    Method based on https://people.sc.fsu.edu/~jburkardt/presentations/truncated_normal.pdf
    """
    def norm_cdf(x):
        # Computes standard normal cumulative distribution function
        return (1. + math.erf(x / math.sqrt(2.))) / 2.

    with torch.no_grad():
        # Values are generated by using a truncated uniform distribution and
        # then using the inverse CDF for the normal distribution.
        # Get upper and lower cdf values
        l = norm_cdf((a - mean) / std)
        u = norm_cdf((b - mean) / std)

        # Uniformly fill tensor with values from [l, u], then translate to
        # [2l-1, 2u-1].
        tensor.uniform_(2 * l - 1, 2 * u - 1)

        # Use inverse cdf transform for normal distribution to get truncated
        # standard normal
        tensor.erfinv_()

        # Transform to proper mean, std
        tensor.mul_(std * math.sqrt(2.))
        tensor.add_(mean)

        # Clamp to ensure it's in the proper range
        tensor.clamp_(min=a, max=b)
        return tensor


def trunc_normal_(tensor, mean=0., std=1., a=-2., b=2.):
    # type: (Tensor, float, float, float, float) -> Tensor
    return _no_grad_trunc_normal_(tensor, mean, std, a, b)

In [ ]:
#| export
def rechunk_zarr(store):
    root_grp = zarr.group(store, overwrite=False)
    for waveform in list(root_grp.array_keys()):
        a = root_grp[waveform]
        da_array = da.from_zarr(a)
        da_array = da_array.rechunk(chunks='auto')
        da_array.to_zarr(url=store, component=waveform.split('_pre_chunk')[0], compute=True)
        del root_grp[waveform]

In [ ]:
#| export
def parallelize_dataframe(df:pd.DataFrame, # Dataframe to parallelize
                          func, # Function to apply
                          n_cores:int # Number of cores
                          )->None:
    df_split = np.array_split(df, n_cores)
    pool = mp.Pool(n_cores)
    pool.map(func, df_split)
    pool.close()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()